## Multi AI Agents - Part 2

- **Create a new multi-agent team that performs creative advertising:**
  - **Define a `Creative_Director` agent that brainstorms 3–5 ad ideas.**
  - **Define a `Strategist` agent that selects the top 2 ideas and explains the reasoning.**
  - **Define a `Copywriter` agent that writes tweets for each top campaign.**
  - **Build a pipeline where the output of each agent is passed to the next, similar to the previous example.**
  - **Try running the full pipeline on a sample prompt (e.g., "Launch campaign for a new eco-friendly water bottle in Bali").**

### Architecture Diagram

```mermaid
flowchart TD
    U([Product / campaign brief]) --> P

    subgraph P["creative_pipeline() + SQLiteSession (shared memory)"]
        direction TB
        D["Creative_Director Agent<br/>out: CreativeIdeas (3-5 ideas)"]
        S["Strategist Agent<br/>out: SelectedCampaigns (top 2 + reasoning)"]
        C["Copywriter Agent<br/>out: TweetCopy (2-3 tweets per campaign)"]
        D -->|ideas, one per line| S
        S -->|top_campaigns, one per line| C
    end

    C --> O["print_markdown()<br/>Top Campaigns + Tweets"]

    SET["Setup: .env / getpass<br/>OPENAI_API_KEY, TAVILY_API_KEY<br/>OpenAI Agents SDK + Pydantic"] -.-> P
```

```text
 Brief -> creative_pipeline() -> Creative_Director -> Strategist -> Copywriter -> Top Campaigns + Tweets
                                 (3-5 ideas)         (picks top 2)  (tweets)
          shared memory: SQLiteSession("creative_pipeline_practice"); LLM = gpt_model
```


### Setup tools and libraries

In [1]:
!pip install -q openai-agents==0.2.2 python-dotenv requests pydantic

In [2]:
import os
import requests
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, SQLiteSession
from typing_extensions import TypedDict
from pydantic import BaseModel

In [3]:
# Setup env. and Load API keys from .env file
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")
    
# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

if not tavily_api_key:
    from getpass import getpass
    tavily_api_key = getpass("Tavily API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key

openai_client = OpenAI(api_key=openai_api_key)

print("✅ API keys loaded")
print(f"  OpenAI … {openai_api_key[:5]}***")
print(f"  Tavily … {tavily_api_key[:5]}***")

✅ API keys loaded
  OpenAI … sk-pr***
  Tavily … tvly-***


In [4]:
gpt_model = "gpt-6.1-terra"

### Setup Tavily Search Tool and Markdown class

In [5]:
# Define a function to display markdown in Jupyter Notebook
def print_markdown(text):
    display(Markdown(text))

### Creative Director Agent

In [6]:
session = SQLiteSession("creative_pipeline_practice")

class CreativeIdeas(BaseModel):
    ideas: list[str]

creative_director = Agent(
    name = "Creative_Director",
    instructions = """
        Context:
        You are a creative director at an advertising agency.

        Instruction:
        Brainstorm 3-5 creative ad campaign ideas for the given product.

        Input:
        The product name or description.

        Output:
        A list of 3-5 creative campaign ideas (as bullet points or short phrases).
        """,
    model = gpt_model,
    output_type = CreativeIdeas
)

### Strategist Agent

In [7]:

class SelectedCampaigns(BaseModel):
    top_campaigns: list[str]
    reasoning: str

strategist = Agent(
    name = "Strategist",
    instructions = """
        Context:
        You are a marketing strategist evaluating campaign ideas.

        Instruction:
        From the provided list of campaign ideas, select the top 2 that are most promising and explain your reasoning.

        Input:
        A list of campaign ideas (one per line).

        Output:
        A list of the top 2 campaign ideas and a paragraph explaining why they stand out.
        """,
    model = gpt_model,
    output_type = SelectedCampaigns,
)


### Copywriter Agent

In [8]:
class TweetCopy(BaseModel):
    tweets: list[str]

copywriter = Agent(
    name = "Copywriter",
    instructions = """
        Context:
        You are a copywriter creating social media content.

        Instruction:
        For each of the top campaign ideas provided, write 2-3 engaging tweets that could be used to promote the campaign.

        Input:
        The selected top campaign ideas (one per line).

        Output:
        A list of 2-3 tweets for each campaign idea.
        """,
    model = gpt_model,
    output_type = TweetCopy)

### Creative Pipeline - Agentic Flow

In [9]:
async def creative_pipeline(product_name: str):
    director_result = await Runner.run(creative_director, product_name, session = session)
    ideas = director_result.final_output.ideas

    strategist_input = "\n".join(ideas)
    strategist_result = await Runner.run(strategist, strategist_input, session = session)
    top_campaigns = strategist_result.final_output.top_campaigns

    tweets_input = "\n".join(top_campaigns)
    copywriter_result = await Runner.run(copywriter, tweets_input, session = session)
    tweets = copywriter_result.final_output.tweets

    print_markdown("### **Top Campaigns:**\n- " + "\n- ".join(top_campaigns))
    print_markdown("### **Tweets:**\n- " + "\n- ".join(tweets))

In [10]:
await creative_pipeline("Launch campaign for a new eco-friendly water bottle in Denmark")

### **Top Campaigns:**
- “Refill Like a Dane”: Celebrate Denmark’s tap-water culture with refill stations and a map of places to top up.
- “Take the Scenic Sip”: Partner with cyclists to share bottle-friendly routes through cities and countryside.

### **Tweets:**
- Refill Like a Dane: Great water is closer than you think. Grab your reusable bottle, explore our refill map, and top up on the go. 💧 #RefillLikeADane
- Refill Like a Dane: From your morning commute to an afternoon by the harbour, keep your bottle close and your next refill closer. Explore the map! 🇩🇰
- Take the Scenic Sip: Two wheels, a reusable bottle, and a route worth taking. Discover cyclist-recommended rides across Denmark. 🚲 #TakeTheScenicSip
- Take the Scenic Sip: City streets or countryside escapes? Wherever you ride, bring water along for the journey. Find your next cycling route. 💧🚲